In [0]:
# Import required PySpark functions
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
# 1. Load Data from Existing Databricks Table

# Read the retail dataset from the Databricks table
df = spark.table("demoworkspacejoby.default.retail_data_analytics")

# Preview the data
df.show(5)

# Check schema
df.printSchema()

+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+
|Invoice|StockCode|         Description|Quantity|        InvoiceDate|Price|Customer ID|       Country|
+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+
| 557112|    23243|SET OF TEA COFFEE...|       1|2011-06-16 16:31:00| 4.95|       NULL|United Kingdom|
| 557112|    23245|SET OF 3 REGENCY ...|       4|2011-06-16 16:31:00|10.79|       NULL|United Kingdom|
| 557112|    23251|VINTAGE RED ENAME...|       2|2011-06-16 16:31:00| 2.46|       NULL|United Kingdom|
| 557112|    23256|CHILDRENS CUTLERY...|       1|2011-06-16 16:31:00| 8.29|       NULL|United Kingdom|
| 557112|    23298|      SPOTTY BUNTING|       2|2011-06-16 16:31:00|10.79|       NULL|United Kingdom|
+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+
only showing top 5 rows
root
 |-- Invoice: string (nullable = true)
 |-- 

In [0]:
# 2. Clean and prepare base dataset
df_base = (
    df
    .dropDuplicates()  # remove duplicate rows
    .filter(col("Customer ID").isNotNull())  # remove null customers
    .filter(col("Price") > 0)  # remove invalid prices
    .withColumn("Customer_ID", col("Customer ID").cast("bigint"))  # standardize column
    .withColumn("InvoiceDate", to_timestamp(col("InvoiceDate")))  # ensure timestamp
    .withColumn("Revenue", col("Quantity") * col("Price"))  # create revenue column
    .withColumn("is_cancelled", when(col("Invoice").startswith("C"), 1).otherwise(0))  # flag cancellations
    .withColumn("invoice_year_month", date_format(col("InvoiceDate"), "yyyyMM"))  # create YYYYMM column
)

df_base.show(5)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+-----------+-------+------------+------------------+
|Invoice|StockCode|         Description|Quantity|        InvoiceDate|Price|Customer ID|       Country|Customer_ID|Revenue|is_cancelled|invoice_year_month|
+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+-----------+-------+------------+------------------+
| 489434|    21871| SAVE THE PLANET MUG|      24|2009-12-01 07:45:00| 1.25|    13085.0|United Kingdom|      13085|   30.0|           0|            200912|
| 489436|    21754|HOME BUILDING BLO...|       3|2009-12-01 09:06:00| 5.95|    13078.0|United Kingdom|      13078|  17.85|           0|            200912|
| 489436|    22109|FULL ENGLISH BREA...|      16|2009-12-01 09:06:00| 3.39|    13078.0|United Kingdom|      13078|  54.24|           0|            200912|
| 489439|    21493| VINTAGE DESIGN G...|      12|2009-12-01 09:28:00| 

In [0]:
# 3. Create Sales-Only Data

# Keep only valid sales (positive quantity)
df_sales = df_base.filter(col("Quantity") > 0)

df_sales.show(5)

+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+-----------+-------+------------+------------------+
|Invoice|StockCode|         Description|Quantity|        InvoiceDate|Price|Customer ID|       Country|Customer_ID|Revenue|is_cancelled|invoice_year_month|
+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+-----------+-------+------------+------------------+
| 489434|    21871| SAVE THE PLANET MUG|      24|2009-12-01 07:45:00| 1.25|    13085.0|United Kingdom|      13085|   30.0|           0|            200912|
| 489436|    21754|HOME BUILDING BLO...|       3|2009-12-01 09:06:00| 5.95|    13078.0|United Kingdom|      13078|  17.85|           0|            200912|
| 489436|    22109|FULL ENGLISH BREA...|      16|2009-12-01 09:06:00| 3.39|    13078.0|United Kingdom|      13078|  54.24|           0|            200912|
| 489439|    21493| VINTAGE DESIGN G...|      12|2009-12-01 09:28:00| 

In [0]:
# 4. Monthly Placed and Canceled Orders

# Get unique invoices
df_orders = df_base.select("Invoice", "invoice_year_month", "is_cancelled").distinct()

# Aggregate orders
monthly_orders_df = (
    df_orders
    .groupBy("invoice_year_month")
    .agg(
        countDistinct("Invoice").alias("total_orders"),
        sum("is_cancelled").alias("canceled_orders")
    )
    # placed orders = total - canceled (correct logic)
    .withColumn("placed_orders", col("total_orders") - col("canceled_orders"))
    .orderBy("invoice_year_month")
)

monthly_orders_df.show()

+------------------+------------+---------------+-------------+
|invoice_year_month|total_orders|canceled_orders|placed_orders|
+------------------+------------+---------------+-------------+
|            200912|        1900|            388|         1512|
|            201001|        1296|            285|         1011|
|            201002|        1333|            229|         1104|
|            201003|        1907|            383|         1524|
|            201004|        1615|            286|         1329|
|            201005|        1768|            391|         1377|
|            201006|        1833|            336|         1497|
|            201007|        1713|            332|         1381|
|            201008|        1547|            254|         1293|
|            201009|        2041|            352|         1689|
|            201010|        2586|            453|         2133|
|            201011|        3145|            558|         2587|
|            201012|        1708|       

In [0]:
# 5. Monthly Sales

# Calculate total sales per month
monthly_sales_df = (
    df_sales
    .groupBy("invoice_year_month")
    .agg(round(sum("Revenue"), 2).alias("monthly_sales"))
    .orderBy("invoice_year_month")
)

monthly_sales_df.show()

+------------------+-------------+
|invoice_year_month|monthly_sales|
+------------------+-------------+
|            200912|    683504.01|
|            201001|    555802.67|
|            201002|    504558.96|
|            201003|    696978.47|
|            201004|     591982.0|
|            201005|    597833.38|
|            201006|    636371.13|
|            201007|    589736.17|
|            201008|     602224.6|
|            201009|    829013.95|
|            201010|   1033112.01|
|            201011|   1166460.02|
|            201012|    570422.73|
|            201101|    568101.31|
|            201102|    446084.92|
|            201103|    594081.76|
|            201104|    468374.33|
|            201105|    677355.15|
|            201106|    660046.05|
|            201107|     598962.9|
+------------------+-------------+
only showing top 20 rows


In [0]:
# 6. Monthly Sales Growth

# Window ordered by time (global time series)
window_month = Window.orderBy("invoice_year_month")

# Calculate month-over-month growth
monthly_sales_growth = (
    monthly_sales_df
    .withColumn(
        "sales_growth_pct",
        (col("monthly_sales") - lag("monthly_sales").over(window_month)) /
        lag("monthly_sales").over(window_month)
    )
)

monthly_sales_growth.show()
monthly_sales.show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+------------------+-------------+--------------------+
|invoice_year_month|monthly_sales|    sales_growth_pct|
+------------------+-------------+--------------------+
|            200912|    683504.01|                NULL|
|            201001|    555802.67|-0.18683334425499562|
|            201002|    504558.96|-0.09219766792412137|
|            201003|    696978.47| 0.38136179367422185|
|            201004|     591982.0|-0.15064521290019184|
|            201005|    597833.38|0.009884388376673624|
|            201006|    636371.13| 0.06446235906064662|
|            201007|    589736.17|-0.07328264561593163|
|            201008|     602224.6| 0.02117629990373481|
|            201009|    829013.95|  0.3765859946604639|
|            201010|   1033112.01| 0.24619375825943587|
|            201011|   1166460.02| 0.12907410688217633|
|            201012|    570422.73| -0.5109796133432846|
|            201101|    568101.31|-0.00406964848683...|
|            201102|    446084.92|-0.21477927942

In [0]:
# 7. Monthly Active Users

# Count unique customers per month
monthly_active_users_df = (
    df_sales
    .groupBy("invoice_year_month")
    .agg(countDistinct("Customer_ID").alias("active_users"))
    .orderBy("invoice_year_month")
)

monthly_active_users_df.show()

+------------------+------------+
|invoice_year_month|active_users|
+------------------+------------+
|            200912|         955|
|            201001|         720|
|            201002|         772|
|            201003|        1057|
|            201004|         942|
|            201005|         966|
|            201006|        1041|
|            201007|         928|
|            201008|         911|
|            201009|        1145|
|            201010|        1497|
|            201011|        1607|
|            201012|         885|
|            201101|         741|
|            201102|         758|
|            201103|         974|
|            201104|         856|
|            201105|        1056|
|            201106|         991|
|            201107|         949|
+------------------+------------+
only showing top 20 rows


In [0]:
# 8. New and Existing Users

# Find first purchase month per customer
first_purchase_df = (
    df_sales
    .groupBy("Customer_ID")
    .agg(min("invoice_year_month").alias("first_purchase_month"))
)

# Join back to label user type
user_tx = df_sales.join(first_purchase_df, on="Customer_ID")

user_tx = user_tx.withColumn(
    "user_type",
    when(col("invoice_year_month") == col("first_purchase_month"), "new")
    .otherwise("existing")
)

# Pivot to get counts
monthly_user_pivot_df = (
    user_tx
    .groupBy("invoice_year_month")
    .pivot("user_type", ["new", "existing"])
    .agg(countDistinct("Customer_ID"))
    .fillna(0)
    .orderBy("invoice_year_month")
)

monthly_user_pivot_df.show()

+------------------+---+--------+
|invoice_year_month|new|existing|
+------------------+---+--------+
|            200912|955|       0|
|            201001|383|     337|
|            201002|374|     398|
|            201003|443|     614|
|            201004|294|     648|
|            201005|254|     712|
|            201006|270|     771|
|            201007|186|     742|
|            201008|162|     749|
|            201009|243|     902|
|            201010|377|    1120|
|            201011|325|    1282|
|            201012| 76|     809|
|            201101| 71|     670|
|            201102|124|     634|
|            201103|179|     795|
|            201104|106|     750|
|            201105|111|     945|
|            201106|108|     883|
|            201107|102|     847|
+------------------+---+--------+
only showing top 20 rows


In [0]:
# 9. RFM Calculation

# Prepare RFM base data
rfm_df = df_sales.withColumn("invoice_amount", col("Revenue"))

# Get reference date (next day of max invoice date)
# Avoid collect() → use aggregation safely
reference_date = (
    rfm_df
    .agg(date_add(to_date(max("InvoiceDate")), 1).alias("ref_date"))
    .first()["ref_date"]
)

# Calculate RFM metrics
rfm_table = (
    rfm_df
    .groupBy("Customer_ID")
    .agg(
        datediff(lit(reference_date), to_date(max("InvoiceDate"))).alias("Recency"),
        countDistinct("Invoice").alias("Frequency"),
        round(sum("invoice_amount"), 2).alias("Monetary")
    )
)

rfm_table.show()

+-----------+-------+---------+---------+
|Customer_ID|Recency|Frequency| Monetary|
+-----------+-------+---------+---------+
|      12980|    158|       21| 16245.78|
|      12924|     89|       12|   2469.4|
|      15380|      9|       11|  3248.96|
|      15196|    433|        2|    313.7|
|      15920|    155|        9|   569.14|
|      17954|      6|       19|  3804.38|
|      16210|      2|       35| 33750.97|
|      17858|      6|       30| 12518.65|
|      13089|      3|      203|113416.91|
|      16641|    106|        2|    554.5|
|      18170|     34|        9|   2878.0|
|      15795|    150|        8|  1979.87|
|      17231|     13|       25|   7064.8|
|      14380|    683|        1|    48.96|
|      17625|     19|        8|   5978.9|
|      16779|      3|       53| 32809.03|
|      12500|     24|       19|  5955.21|
|      14030|     19|       20|   7521.8|
|      17287|     27|       21|  3826.86|
|      18092|      3|       18| 14693.72|
+-----------+-------+---------+---

In [0]:
# 10. RFM Scoring

# Define windows for scoring
window_r = Window.orderBy(col("Recency").asc())
window_f = Window.orderBy(col("Frequency").asc())
window_m = Window.orderBy(col("Monetary").asc())

# Assign scores (1–5 scale)
rfm_metrics = (
    rfm_table
    .withColumn("R_score", 6 - ntile(5).over(window_r))  # lower recency → higher score
    .withColumn("F_score", ntile(5).over(window_f))
    .withColumn("M_score", ntile(5).over(window_m))
)

rfm_metrics.show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-----------+-------+---------+--------+-------+-------+-------+
|Customer_ID|Recency|Frequency|Monetary|R_score|F_score|M_score|
+-----------+-------+---------+--------+-------+-------+-------+
|      14095|    723|        1|    2.95|      1|      2|      1|
|      16738|    298|        1|    3.75|      2|      1|      1|
|      13788|    506|        1|    3.75|      1|      1|      1|
|      14792|     64|        1|     6.2|      3|      1|      1|
|      15913|    535|        1|     6.3|      1|      2|      1|
|      15040|    542|        1|    7.49|      1|      2|      1|
|      18115|    698|        1|     9.7|      1|      2|      1|
|      17378|    375|        1|   10.95|      2|      1|      1|
|      17956|    250|        1|   12.75|      2|      1|      1|
|      16878|     85|        1|    13.3|      3|      1|      1|
|      14900|    523|        1|   13.92|      1|      1|      1|
|      14580|    519|        1|   14.85|      1|      1|      1|
|      12846|    683|    

In [0]:
# 11. Customer Segmentation

# Combine R and F scores
rfm_metrics = rfm_metrics.withColumn(
    "RF_score",
    concat(col("R_score"), col("F_score"))
)

# Assign segment labels
rfm_metrics = rfm_metrics.withColumn(
    "Segment",
    when(col("RF_score").rlike("^[1-2][1-2]$"), "Hibernating")
    .when(col("RF_score").rlike("^[1-2][3-4]$"), "At Risk")
    .when(col("RF_score").rlike("^[1-2]5$"), "Can't Lose")
    .when(col("RF_score").rlike("^3[1-2]$"), "About to Sleep")
    .when(col("RF_score").rlike("^33$"), "Need Attention")
    .when(col("RF_score").rlike("^[3-4][4-5]$"), "Loyal Customers")
    .when(col("RF_score").rlike("^41$"), "Promising")
    .when(col("RF_score").rlike("^51$"), "New Customers")
    .when(col("RF_score").rlike("^[4-5][2-3]$"), "Potential Loyalists")
    .when(col("RF_score").rlike("^5[4-5]$"), "Champions")
)

rfm_metrics.select("Customer_ID", "Recency", "Frequency", "Monetary", "Segment").show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-----------+-------+---------+--------+-------------+
|Customer_ID|Recency|Frequency|Monetary|      Segment|
+-----------+-------+---------+--------+-------------+
|      12713|      1|        1|  848.55|New Customers|
|      13436|      2|        1|  196.89|New Customers|
|      15520|      2|        1|   343.5|New Customers|
|      13298|      2|        1|   360.0|New Customers|
|      15195|      3|        1|  3861.0|New Customers|
|      14578|      4|        1|  168.63|New Customers|
|      12478|      4|        1|  680.99|New Customers|
|      16528|      4|        1|  244.41|New Customers|
|      15318|      4|        1|  312.62|New Customers|
|      12442|      4|        1|  172.06|New Customers|
|      12650|      4|        1|  314.44|New Customers|
|      17914|      4|        1|  329.36|New Customers|
|      13790|      5|        1|   348.8|New Customers|
|      16597|      5|        1|   90.04|New Customers|
|      18015|      5|        1|  120.03|New Customers|
|      123

In [0]:
# 12. Segment Summary

# Summary statistics per segment
segment_summary = (
    rfm_metrics
    .groupBy("Segment")
    .agg(
        round(avg("Recency")).alias("Recency_mean"),
        count("*").alias("Customer_count"),
        round(avg("Frequency")).alias("Frequency_mean"),
        round(avg("Monetary")).alias("Monetary_mean")
    )
)

segment_summary.show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-------------------+------------+--------------+--------------+-------------+
|            Segment|Recency_mean|Customer_count|Frequency_mean|Monetary_mean|
+-------------------+------------+--------------+--------------+-------------+
|      New Customers|        11.0|            71|           1.0|        351.0|
|          Promising|        38.0|           163|           1.0|        335.0|
|     About to Sleep|       108.0|           422|           1.0|        533.0|
|        Hibernating|       445.0|          1429|           1.0|        395.0|
|Potential Loyalists|        26.0|           711|           3.0|       1220.0|
|            At Risk|       407.0|           831|           4.0|       1256.0|
|     Need Attention|       109.0|           266|           3.0|       1352.0|
|    Loyal Customers|        68.0|          1079|          10.0|       4320.0|
|          Champions|         9.0|           816|          20.0|      10953.0|
|         Can't Lose|       338.0|            90|   

In [0]:
# 13. Total revenue per segment
segment_revenue = (
    rfm_metrics
    .groupBy("Segment")
    .agg(round(sum("Monetary"), 2).alias("total_revenue"))
)

# Calculate total revenue
total_revenue = segment_revenue.agg(sum("total_revenue")).first()[0]

# Calculate percentage contribution
segment_revenue = segment_revenue.withColumn(
    "revenue_pct",
    round(col("total_revenue") / total_revenue * 100, 2)
)

segment_revenue.orderBy(col("total_revenue").desc()).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-------------------+-------------+-----------+
|            Segment|total_revenue|revenue_pct|
+-------------------+-------------+-----------+
|          Champions|   8937954.57|      51.44|
|    Loyal Customers|   4660777.96|      26.82|
|            At Risk|   1043968.62|       6.01|
|Potential Loyalists|    867483.92|       4.99|
|         Can't Lose|    635833.47|       3.66|
|        Hibernating|     564879.3|       3.25|
|     Need Attention|    359575.81|       2.07|
|     About to Sleep|    224765.33|       1.29|
|          Promising|     54664.96|       0.31|
|      New Customers|     24900.31|       0.14|
+-------------------+-------------+-----------+

